# SWAT+ Channel Output Extraction

Extracts raw SWAT+ model output (`channel_sdmorph_day.csv`, etc.) for
selected channels and builds daily/monthly/long-term-monthly/seasonal/annual
flow summaries, for all three scenario groups in the study design:

1. **Historical baseline** — one run, full record, no future periods.
2. **Climate-change-only** — 2010 baseline land use, 8 climate scenarios
   (2 SSPs x 4 GCM archetypes), each a single continuous 2015–2100 run split
   here into near/mid/far future periods.
3. **Land-use + climate-change (combined)** — same 8 climate scenarios (plus
   historical) re-run under each of the 3 future land-use maps
   (`2040_lulc_model` / `2065_lulc_model` / `2090_lulc_model`, corresponding to
   near/mid/far future respectively) — no period split needed here since the
   LULC folder itself already picks out the period.

Shared extraction logic (copy required files, load + clean the daily channel
CSV, build daily/monthly/seasonal/annual pivots) lives in
`swat_extraction_utils.py` 

**Writes:** `SWAT_Outputs/{model_scenario}/{inside_scenario}/Channel_combined/`
(+ `/near_future`, `/mid_future`, `/far_future` subfolders for group 2).

**Consumed by:** `10_SWAT_output_aggregation.ipynb`,

In [ ]:
import os

import pandas as pd

from swat_extraction_utils import process_scenario


**Channel reference:**
* 53 - marikhola_station
* 98 - bagaswotigau_station
* 69 - jalakundi_station
* 12, 33 - madi - dang diversion
* 102 - Naumure dam axis
* 100 - Lamatal re-regulating (Kapilvastu diversion)
* 97 - Praganna badkapath
* 50 - Sikta HW

## Config (shared across all three scenario groups)

In [ ]:
CHANNELS_TO_EXTRACT = [53, 98, 69, 12, 33, 100, 102, 97, 50]


## 1. Historical baseline

Source scenario is `Calibrated_model` -- the calibrated snapshot of the
QSWAT+ project's base run. Output is
written to `historical_climate/`

In [ ]:
process_scenario(
    model_scenario='2010_lulc_Historical BASELINE model',
    inside_scenario='Calibrated_model',
    channels=CHANNELS_TO_EXTRACT,
    output_name='historical_climate',
)


## 2. Climate-change-only scenarios

2010 baseline land use, one continuous 2015–2100 run per climate scenario
-- split into near/mid/far future periods here since a single run spans
all three.

In [ ]:
FUTURE_SCENARIOS = os.listdir('../All_DATA/Climate/New_Future_Climate/SWAT_input/')

# End=None means "open ended" -- runs to whatever the last available date in
# the data is (far future scenarios don't all stop on the same date).
PERIODS = [
    ('near_future', pd.Timestamp('2026-01-01'), pd.Timestamp('2050-12-31')),
    ('mid_future', pd.Timestamp('2051-01-01'), pd.Timestamp('2075-12-31')),
    ('far_future', pd.Timestamp('2076-01-01'), None),
]

for inside_scenario in FUTURE_SCENARIOS:
    process_scenario(
        model_scenario='2010_lulc_Historical BASELINE model',
        inside_scenario=inside_scenario,
        channels=CHANNELS_TO_EXTRACT,
        periods=PERIODS,
        round_to=2,
    )


## 3. Land-use + climate-change (combined) scenarios

Same 8 climate scenarios (plus historical) re-run under each future land-use
map. No period split needed -- the LULC folder (`2040_lulc_model` = near,
`2065_lulc_model` = mid, `2090_lulc_model` = far) already picks out the
period, matching what `10_SWAT_output_aggregation.ipynb` expects.

Previously this only processed one hardcoded `MODEL_SCENARIO` at a time,
requiring you to hand-edit and re-run it once per LULC year -- now it loops
over all three automatically.

In [ ]:
FUTURE_LULC_MODELS = ['2040_lulc_model', '2065_lulc_model', '2090_lulc_model']
INSIDE_SCENARIOS = ['historical_climate'] + FUTURE_SCENARIOS

for model_scenario in FUTURE_LULC_MODELS:
    for inside_scenario in INSIDE_SCENARIOS:
        try:
            process_scenario(
                model_scenario=model_scenario,
                inside_scenario=inside_scenario,
                channels=CHANNELS_TO_EXTRACT,
                round_to=3,
            )
        except FileNotFoundError as e:
            print(f"Skipping '{model_scenario}/{inside_scenario}': {e}")
